In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Football'}, config=config1)

{'topic': 'Football',
 'joke': 'Why did the football coach go to the bank?\n\nTo get his quarter-back!',
 'explanation': 'This joke is a classic **play on words**, specifically using a **homophone** (a word that sounds the same but has different meanings).\n\nHere\'s the breakdown:\n\n1.  **Football Meaning:**\n    *   In American football, a **quarterback** is a very important player position, often the leader of the offense who throws the ball.\n    *   A coach would, metaphorically, want to "get his quarterback" – meaning, have him ready, on the field, performing well, or even recruit him.\n\n2.  **Bank/Money Meaning:**\n    *   When you break down the sound of "quarterback," it can also sound like "quarter" (a 25-cent coin) and "back" (as in to retrieve something, or get money back).\n    *   A bank is where you go to get money, including coins like quarters.\n\n**The Humor:**\n\nThe joke sets you up to think in a football context (coach, football). Your brain expects a football-re

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football coach go to the bank?\n\nTo get his quarter-back!', 'explanation': 'This joke is a classic **play on words**, specifically using a **homophone** (a word that sounds the same but has different meanings).\n\nHere\'s the breakdown:\n\n1.  **Football Meaning:**\n    *   In American football, a **quarterback** is a very important player position, often the leader of the offense who throws the ball.\n    *   A coach would, metaphorically, want to "get his quarterback" – meaning, have him ready, on the field, performing well, or even recruit him.\n\n2.  **Bank/Money Meaning:**\n    *   When you break down the sound of "quarterback," it can also sound like "quarter" (a 25-cent coin) and "back" (as in to retrieve something, or get money back).\n    *   A bank is where you go to get money, including coins like quarters.\n\n**The Humor:**\n\nThe joke sets you up to think in a football context (coach, football). Your brain ex

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football coach go to the bank?\n\nTo get his quarter-back!', 'explanation': 'This joke is a classic **play on words**, specifically using a **homophone** (a word that sounds the same but has different meanings).\n\nHere\'s the breakdown:\n\n1.  **Football Meaning:**\n    *   In American football, a **quarterback** is a very important player position, often the leader of the offense who throws the ball.\n    *   A coach would, metaphorically, want to "get his quarterback" – meaning, have him ready, on the field, performing well, or even recruit him.\n\n2.  **Bank/Money Meaning:**\n    *   When you break down the sound of "quarterback," it can also sound like "quarter" (a 25-cent coin) and "back" (as in to retrieve something, or get money back).\n    *   A bank is where you go to get money, including coins like quarters.\n\n**The Humor:**\n\nThe joke sets you up to think in a football context (coach, football). Your brain e

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)

{'topic': 'Cricket',
 'joke': 'Why did the cricket player bring a dictionary to the game?\n\nBecause he kept getting **stumped**!',
 'explanation': 'This joke is a classic pun that plays on the two different meanings of the word "stumped":\n\n1.  **In Cricket:** To be "stumped" is a specific way for a batsman to be dismissed (get out). It happens when the batsman is out of their crease (the safe area in front of the wicket) and the wicketkeeper dislodges the bails (the small pieces of wood on top of the stumps) with the ball. So, if a player "kept getting stumped," it means they kept getting out in this particular way.\n\n2.  **General English:** To be "stumped" means to be perplexed, confused, or unable to answer a question or solve a problem. If you\'re "stumped" by a word, you might look it up in a dictionary.\n\nThe humor comes from the joke pretending the cricket player is confused by the *word* "stumped" (meaning he\'s perplexed and needs a dictionary), when in reality, he\'s jus

In [12]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket player bring a dictionary to the game?\n\nBecause he kept getting **stumped**!', 'explanation': 'This joke is a classic pun that plays on the two different meanings of the word "stumped":\n\n1.  **In Cricket:** To be "stumped" is a specific way for a batsman to be dismissed (get out). It happens when the batsman is out of their crease (the safe area in front of the wicket) and the wicketkeeper dislodges the bails (the small pieces of wood on top of the stumps) with the ball. So, if a player "kept getting stumped," it means they kept getting out in this particular way.\n\n2.  **General English:** To be "stumped" means to be perplexed, confused, or unable to answer a question or solve a problem. If you\'re "stumped" by a word, you might look it up in a dictionary.\n\nThe humor comes from the joke pretending the cricket player is confused by the *word* "stumped" (meaning he\'s perplexed and needs a dictionary), when in

In [13]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket player bring a dictionary to the game?\n\nBecause he kept getting **stumped**!', 'explanation': 'This joke is a classic pun that plays on the two different meanings of the word "stumped":\n\n1.  **In Cricket:** To be "stumped" is a specific way for a batsman to be dismissed (get out). It happens when the batsman is out of their crease (the safe area in front of the wicket) and the wicketkeeper dislodges the bails (the small pieces of wood on top of the stumps) with the ball. So, if a player "kept getting stumped," it means they kept getting out in this particular way.\n\n2.  **General English:** To be "stumped" means to be perplexed, confused, or unable to answer a question or solve a problem. If you\'re "stumped" by a word, you might look it up in a dictionary.\n\nThe humor comes from the joke pretending the cricket player is confused by the *word* "stumped" (meaning he\'s perplexed and needs a dictionary), when i

### Time Travel

In [14]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f15d969-8ed6-683c-8000-0c7a2f030ccd'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [16]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd"}})

EmptyInputError: Received no input for __start__

In [20]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket player bring a dictionary to the game?\n\nBecause he kept getting **stumped**!', 'explanation': 'This joke is a classic pun that plays on the two different meanings of the word "stumped":\n\n1.  **In Cricket:** To be "stumped" is a specific way for a batsman to be dismissed (get out). It happens when the batsman is out of their crease (the safe area in front of the wicket) and the wicketkeeper dislodges the bails (the small pieces of wood on top of the stumps) with the ball. So, if a player "kept getting stumped," it means they kept getting out in this particular way.\n\n2.  **General English:** To be "stumped" means to be perplexed, confused, or unable to answer a question or solve a problem. If you\'re "stumped" by a word, you might look it up in a dictionary.\n\nThe humor comes from the joke pretending the cricket player is confused by the *word* "stumped" (meaning he\'s perplexed and needs a dictionary), when i

#### Updating State

In [21]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd", "checkpoint_ns": ""}}, {'topic':'Tenis'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1dad87-db7a-69e4-8000-526c4a3c881b'}}

In [22]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tenis'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1dad87-db7a-69e4-8000-526c4a3c881b'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-11-07T18:24:38.217571+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15d969-8ed6-683c-8000-0c7a2f030ccd'}}, tasks=(PregelTask(id='fa535102-c62e-d968-e982-5d6974cab634', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket player bring a dictionary to the game?\n\nBecause he kept getting **stumped**!', 'explanation': 'This joke is a classic pun that plays on the two different meanings of the word "stumped":\n\n1.  **In Cricket:** To be "stumped" is a specific way for a batsman to be dismissed (get out). It happens when the bat

### Fault Tolerance

In [23]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [24]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [25]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [26]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [27]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed


In [28]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [29]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1dad89-96f4-661e-8003-9f5518c4550d'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-11-07T18:25:24.719362+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1dad89-96f4-661d-8002-4248156b57b2'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1dad89-96f4-661d-8002-4248156b57b2'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-11-07T18:25:24.719362+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1dad88-78d8-630c-8001-c63c9c4d4164'}}, tasks=(PregelTask(id='956cc831-3cea-8d7a-8c6f-8e9e6c3a77b